# Clay Provenance Analysis — Complete Pipeline (Kaggle Edition)**Matplotlib-only pipeline.** Reads the 18-sheet Excel dataset directly, producesall figures and statistical tables without external plotting dependencies beyondmatplotlib.*Jackson, Majumder, Jaiswal, Patterson & Goswami (2026) | Editor: Sania Mughal*

## 1. Setup & Imports

In [ ]:
import pandas as pdimport numpy as npimport matplotlib.pyplot as pltimport matplotlib.patches as mpatchesimport matplotlib.colors as mcolorsfrom matplotlib.patches import Ellipsefrom scipy.stats import f_oneway, kruskalimport warningswarnings.filterwarnings('ignore')plt.rcParams.update({    'font.family': 'serif',    'font.size': 10,    'axes.titlesize': 13,    'axes.labelsize': 11,    'figure.dpi': 150,    'axes.spines.top': False,    'axes.spines.right': False,})# Colour palette for the 6 clay-source groupsPALETTE = ['#2E86AB', '#E8630A', '#5FAD56', '#9B2335', '#7B68EE', '#FF6F61']print('Setup complete — matplotlib', plt.matplotlib.__version__)

## 2. Load DataAll 18 sheets are read into a dictionary.  The grouping column **`Clay_Type`**(present on most sheets) or **`Assigned_Clay_Source`** (Sheet 16) serves as theprovenance classifier throughout.

In [ ]:
# =====================================================================# >>> SET YOUR FILE PATH HERE <<<# =====================================================================XLSX_PATH = '/kaggle/input/historical-dataset-1/Geochemical_Differentiation_of_Alluvial_vs__Residual_Secondary_Clay_Deposits_Dataset.xlsx'xl  = pd.ExcelFile(XLSX_PATH)ALL = {name: pd.read_excel(xl, sheet_name=name) for name in xl.sheet_names}xl.close()# ── Assign friendly names ────────────────────────────────────────────xrf       = ALL['1_Major_Oxide_Geochem']trace     = ALL['2_Trace_Element_Anal']ree       = ALL['3_REE_Profiles']xrd       = ALL['4_Clay_Mineralogy_XRD']petro     = ALL['5_Petrographic_Anal']ceramic   = ALL['6_Ceramic_Sherd_Chem']indices   = ALL['7_Weathering_Indices']physical  = ALL['8_Physical_Properties']naa       = ALL['9_NAA_Results']xrf_r     = ALL['10_XRF_Results']icp       = ALL['11_ICP_MS_Results']pca       = ALL['12_PCA_Scores']dfa       = ALL['13_DFA_Classification']catchment = ALL['14_Site_Catchment']production= ALL['15_Production_Org']provenance= ALL['16_Provenance_Assign']firing    = ALL['17_Firing_Temp_Est']grain     = ALL['18_Grain_Size_Distrib']# ── Group column & colour map ────────────────────────────────────────GRP = 'Clay_Type'                       # present on most sheetsGRP_PROV = 'Assigned_Clay_Source'       # on provenance sheetgroups   = sorted(xrf[GRP].dropna().unique())GC       = {g: PALETTE[i % len(PALETTE)] for i, g in enumerate(groups)}GS       = {g: g.replace('_', ' ') for g in groups}    # display labelsprint(f'Loaded {len(ALL)} sheets  |  {sum(d.shape[0] for d in ALL.values()):,} total rows')print(f'Groups ({len(groups)}): {list(GS.values())}')print()for name, df in ALL.items():    print(f'  {name:30s}  {df.shape[0]:>5,} rows × {df.shape[1]:>2} cols')

## 3. Oxide Columns & Derived Ratios

In [ ]:
# ── Major oxide columns (Sheet 1) ─────────────────────────────────────oxide_cols = ['SiO2_wt%','Al2O3_wt%','Fe2O3_wt%','MgO_wt%','CaO_wt%',              'Na2O_wt%','K2O_wt%','TiO2_wt%','P2O5_wt%','MnO_wt%']# ── Weathering / provenance indices (Sheet 7) ────────────────────────CIA_col  = 'CIA'ICV_col  = 'ICV'SiAl     = 'SiO2_Al2O3'KNa      = 'K2O_Na2O'FeAl     = 'Fe2O3_MgO'      # actually Fe2O3/MgO in this datasetLaSc     = 'La_Sc'ThSc     = 'Th_Sc'# ── REE columns (Sheet 3) ────────────────────────────────────────────ree_elements = ['La','Ce','Pr','Nd','Sm','Eu','Gd','Tb','Dy','Ho','Er','Tm','Yb','Lu']ree_ppm_cols = [f'{e}_ppm' for e in ree_elements]# Chondrite normalisation values (McDonough & Sun 1995)chondrite_vals = {'La':0.237,'Ce':0.613,'Pr':0.0928,'Nd':0.457,'Sm':0.148,                  'Eu':0.0563,'Gd':0.199,'Tb':0.0361,'Dy':0.246,'Ho':0.0546,                  'Er':0.160,'Tm':0.0247,'Yb':0.161,'Lu':0.0246}# Compute chondrite-normalised columnsfor e in ree_elements:    col = f'{e}_ppm'    ree[f'{e}N'] = ree[col] / chondrite_vals[e]chond_cols = [f'{e}N' for e in ree_elements]# ── PCA / DFA columns ────────────────────────────────────────────────PC1, PC2     = 'PC1_Score', 'PC2_Score'PC1v, PC2v   = 'PC1_Var_%', 'PC2_Var_%'LD1, LD2     = 'DF1_Score', 'DF2_Score'Sil          = 'Silhouette_Coeff'# ── Grain size columns ────────────────────────────────────────────────ClayF  = 'Clay_pct'SiltF  = 'Fine_Silt_pct'FSand  = 'Fine_Sand_pct'FTemp  = 'Est_Firing_Temp_C'print('Column mapping complete.')print(f'  Oxides: {len(oxide_cols)}   REE: {len(ree_ppm_cols)}   Chondrite-norm: {len(chond_cols)}')

## 4. Major Element Analysis

In [ ]:
# ── Table 1: Major Element Statistics ──────────────────────────────────rows = []for g in groups:    m   = xrf[GRP] == g    row = {'Group': GS[g], 'n': m.sum()}    for ox in oxide_cols:        v = xrf.loc[m, ox].dropna()        row[ox] = f'{v.mean():.2f} ± {v.std():.2f}' if len(v) else 'N/A'    rows.append(row)tbl1 = pd.DataFrame(rows)print('Table 1. Major Element Statistics (mean ± σ)')display(tbl1)# ── Figure 1: Major Element Box Plots ─────────────────────────────────n_ox = min(len(oxide_cols), 10)nc, nr = 5, 2fig, axes = plt.subplots(nr, nc, figsize=(18, 10))fig.suptitle('Figure 1. Major Element Distributions by Clay Type',             fontsize=14, fontweight='bold', y=1.02)for i in range(nr * nc):    ax = axes[i // nc, i % nc]    if i < n_ox:        ox   = oxide_cols[i]        data = [xrf.loc[xrf[GRP] == g, ox].dropna() for g in groups]        bp   = ax.boxplot(data, patch_artist=True, widths=0.6,                          medianprops=dict(color='black', lw=1.5),                          flierprops=dict(ms=3))        for patch, colour in zip(bp['boxes'], GC.values()):            patch.set_facecolor(colour)            patch.set_alpha(0.7)        ax.set_ylabel(ox.replace('_wt%', ' wt%'), fontsize=8)        ax.set_xticklabels([GS[g][:8] for g in groups], fontsize=6, rotation=45, ha='right')    else:        ax.set_visible(False)plt.tight_layout()plt.show()

In [ ]:
# ── Figure 2: CIA vs. ICV Scatter ──────────────────────────────────────fig, ax = plt.subplots(figsize=(10, 8))for g, c in GC.items():    m = indices[GRP] == g    ax.scatter(indices.loc[m, CIA_col], indices.loc[m, ICV_col],               c=c, label=GS[g], alpha=0.55, s=40, edgecolors='gray', lw=0.3)ax.axhline(y=1, color='gray', ls='--', alpha=0.5, label='ICV = 1')ax.set_xlabel('CIA (Chemical Index of Alteration)')ax.set_ylabel('ICV (Index of Compositional Variability)')ax.set_title('Figure 2. CIA vs. ICV', fontsize=13, fontweight='bold')ax.legend(fontsize=9, framealpha=0.8)ax.grid(True, alpha=0.15)plt.tight_layout()plt.show()print('\nTable 2. CIA / ICV by Group')for g in groups:    m  = indices[GRP] == g    ci = indices.loc[m, CIA_col]    ic = indices.loc[m, ICV_col]    print(f'  {GS[g]:25s}  CIA = {ci.mean():.1f} ± {ci.std():.1f}'          f'   ICV = {ic.mean():.3f} ± {ic.std():.3f}')

In [ ]:
# ── Figure 12 & 17: Oxide Ratio Biplots ───────────────────────────────fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))for g, c in GC.items():    m = indices[GRP] == g    a1.scatter(indices.loc[m, SiAl], indices.loc[m, FeAl],               c=c, alpha=0.5, s=40, edgecolors='gray', lw=0.3, label=GS[g])    a2.scatter(indices.loc[m, SiAl], indices.loc[m, KNa],               c=c, alpha=0.5, s=40, edgecolors='gray', lw=0.3, label=GS[g])a1.set_xlabel('SiO₂/Al₂O₃');  a1.set_ylabel('Fe₂O₃/MgO')a1.set_title('Fig 12. SiO₂/Al₂O₃ vs Fe₂O₃/MgO', fontweight='bold')a1.legend(fontsize=8); a1.grid(True, alpha=0.15)a2.set_xlabel('SiO₂/Al₂O₃');  a2.set_ylabel('K₂O/Na₂O')a2.set_title('Fig 17. SiO₂/Al₂O₃ vs K₂O/Na₂O', fontweight='bold')a2.legend(fontsize=8); a2.grid(True, alpha=0.15)plt.tight_layout()plt.show()# ── Figure 16: Correlation Matrix (pure matplotlib) ───────────────────avail = [c for c in oxide_cols if c in xrf.columns]corr  = xrf[avail].corr()fig, ax = plt.subplots(figsize=(11, 9))im = ax.imshow(corr.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')labels = [c.replace('_wt%', '') for c in avail]ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha='right', fontsize=9)ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels, fontsize=9)for i in range(len(labels)):    for j in range(len(labels)):        val = corr.values[i, j]        colour = 'white' if abs(val) > 0.6 else 'black'        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=colour)fig.colorbar(im, ax=ax, shrink=0.8, label='Pearson r')ax.set_title('Fig 16. Major Oxide Correlation Matrix', fontsize=13, fontweight='bold')plt.tight_layout()plt.show()

## 5. REE Fingerprinting

In [ ]:
# ── Figure 3: Chondrite-Normalised REE Spider Diagram ─────────────────fig, ax = plt.subplots(figsize=(13, 7))x = np.arange(len(chond_cols))for g, c in GC.items():    d  = ree.loc[ree[GRP] == g, chond_cols].apply(pd.to_numeric, errors='coerce')    mu = d.mean()    sd = d.std()    ax.plot(x, mu, color=c, lw=2.5, label=GS[g], marker='o', ms=5)    ax.fill_between(x, (mu - sd).clip(lower=0.1), mu + sd, color=c, alpha=0.10)ax.set_yscale('log')ax.set_xticks(x)ax.set_xticklabels(ree_elements, fontsize=10)ax.set_ylabel('Sample / Chondrite')ax.set_title('Figure 3. Chondrite-Normalised REE Patterns (Mean ± 1σ)',             fontsize=13, fontweight='bold')ax.legend(fontsize=10)ax.grid(True, alpha=0.3, which='both')plt.tight_layout()plt.show()

In [ ]:
# ── Figure 7: Eu/Eu* vs LREE/HREE ─────────────────────────────────────fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))for g, c in GC.items():    m = ree[GRP] == g    a1.scatter(ree.loc[m, 'LaN_YbN'], ree.loc[m, 'Eu_Eu_star'],               c=c, alpha=0.5, s=40, edgecolors='gray', lw=0.3, label=GS[g])    a2.scatter(ree.loc[m, 'LREE_HREE'], ree.loc[m, 'Eu_Eu_star'],               c=c, alpha=0.5, s=40, edgecolors='gray', lw=0.3, label=GS[g])a1.axhline(y=1, color='gray', ls='--', alpha=0.4)a1.set_xlabel('LaN/YbN');  a1.set_ylabel('Eu/Eu*')a1.set_title('Fig 7. LaN/YbN vs Eu/Eu*', fontweight='bold')a1.legend(fontsize=8);     a1.grid(True, alpha=0.15)a2.axhline(y=1, color='gray', ls='--', alpha=0.4)a2.set_xlabel('LREE/HREE'); a2.set_ylabel('Eu/Eu*')a2.set_title('Fig 14. LREE/HREE vs Eu/Eu*', fontweight='bold')a2.legend(fontsize=8);      a2.grid(True, alpha=0.15)plt.tight_layout()plt.show()# ── Table 3: REE Summary ─────────────────────────────────────────────print('\nTable 3. REE Summary by Group')for g in groups:    m  = ree[GRP] == g    sr = ree.loc[m, 'Total_REE_ppm']    lh = ree.loc[m, 'LREE_HREE']    eu = ree.loc[m, 'Eu_Eu_star']    print(f'  {GS[g]:25s}  ΣREE = {sr.mean():.1f} ± {sr.std():.1f}'          f'   LREE/HREE = {lh.mean():.2f} ± {lh.std():.2f}'          f'   Eu/Eu* = {eu.mean():.3f} ± {eu.std():.3f}')

## 6. Petrography, Grain Size & Firing Temperature

In [ ]:
# ── Figure 8: Fabric Group Distribution per Clay Type ─────────────────
# Petrographic sheet lacks Clay_Type — build Site_Code → Clay_Type lookup
_lookup = xrf.groupby('Site_Code')[[GRP]].agg(lambda x: x.mode().iloc[0]).reset_index()
petro_m = petro.merge(_lookup, on='Site_Code', how='left')

fab_grps = petro['Fabric_Group'].value_counts().index.tolist()
mc = ['#FFD700','#FF6347','#4682B4','#87CEEB','#B0C4DE',
      '#90EE90','#DDA0DD','#FF8C00','#8B4513','#2F4F4F']

fig, ax = plt.subplots(figsize=(13, 7))
y     = np.arange(len(groups))
lefts = np.zeros(len(groups))

for j, fg in enumerate(fab_grps):
    vals = []
    for g in groups:
        total = (petro_m[GRP] == g).sum()
        count = ((petro_m[GRP] == g) & (petro_m['Fabric_Group'] == fg)).sum()
        vals.append(count / total * 100 if total else 0)
    ax.barh(y, vals, left=lefts, height=0.6,
            color=mc[j % len(mc)], edgecolor='white', lw=0.5, label=fg)
    for ii, (v, l) in enumerate(zip(vals, lefts)):
        if v > 5:
            ax.text(l + v/2, ii, f'{v:.0f}%', ha='center', va='center',
                    fontsize=6, fontweight='bold',
                    color='white' if v > 10 else 'black')
    lefts += vals

ax.set_yticks(y)
ax.set_yticklabels([GS[g] for g in groups], fontsize=10)
ax.set_xlabel('% of Specimens')
ax.set_title('Fig 8. Fabric Group Distribution by Clay Type',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=7, loc='center right', bbox_to_anchor=(1.35, 0.5))
ax.set_xlim(0, 105)
plt.tight_layout()
plt.show()

In [ ]:
# ── Figure 9: Grain Size (Clay % vs Silt %) ──────────────────────────
fig, ax = plt.subplots(figsize=(10, 8))
for g, c in GC.items():
    m  = grain[GRP] == g
    sz = grain.loc[m, FSand].clip(lower=1) * 3
    ax.scatter(grain.loc[m, ClayF], grain.loc[m, SiltF],
               s=sz, c=c, alpha=0.45, edgecolors='gray', lw=0.3, label=GS[g])

ax.set_xlabel('Clay %  (< 2 µm)')
ax.set_ylabel('Fine Silt %  (2–20 µm)')
ax.set_title('Figure 9. Grain Size — Clay vs Fine Silt',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.15)
plt.tight_layout()
plt.show()

# ── Figure 10: Firing Temperature Violin Plot ─────────────────────────
# Build lookup for Clay_Type from xrf Site_Code mapping
fire_m = firing.merge(_lookup, on='Site_Code', how='left')

fig, ax = plt.subplots(figsize=(10, 7))
fdata = [fire_m.loc[fire_m[GRP] == g, FTemp].dropna().values for g in groups]

parts = ax.violinplot(fdata, positions=range(1, len(groups)+1),
                      showmeans=True, showextrema=True, showmedians=True)
for i, pc in enumerate(parts['bodies']):
    pc.set_facecolor(list(GC.values())[i])
    pc.set_alpha(0.55)
parts['cmeans'].set_color('black')
parts['cmedians'].set_color('red')

ax.set_xticks(range(1, len(groups)+1))
ax.set_xticklabels([GS[g][:10] for g in groups], fontsize=7, rotation=45, ha='right')
ax.set_ylabel('Estimated Firing Temperature (°C)')
ax.set_title('Figure 10. Firing Temperature Distributions',
             fontsize=13, fontweight='bold')
ax.grid(True, alpha=0.15, axis='y')
plt.tight_layout()
plt.show()

## 7. Multivariate Statistics (PCA, DFA, La/Sc vs Th/Sc)

In [ ]:
# ── Figure 5: PCA Biplot with 95 % Confidence Ellipses ────────────────fig, ax = plt.subplots(figsize=(10, 8))for g, c in GC.items():    m   = pca[GRP] == g    sub = pca.loc[m, [PC1, PC2]].dropna()    ax.scatter(sub[PC1], sub[PC2], c=c, alpha=0.5, s=45,               edgecolors='gray', lw=0.3, label=GS[g])    if len(sub) > 2:        mx, my = sub[PC1].mean(), sub[PC2].mean()        cov = np.cov(sub[[PC1, PC2]].values.T)        ev, evc = np.linalg.eigh(cov)        order   = ev.argsort()[::-1]        ev, evc = ev[order], evc[:, order]        angle = np.degrees(np.arctan2(evc[1, 0], evc[0, 0]))        w, h  = 2 * np.sqrt(5.991 * ev)        # chi² 95 %        ell   = Ellipse(xy=(mx, my), width=w, height=h, angle=angle,                        fc=c, alpha=0.08, ec=c, lw=1.5)        ax.add_patch(ell)# Axis labels with variance explainedv1 = pca[PC1v].median()v2 = pca[PC2v].median()ax.set_xlabel(f'PC 1  ({v1:.1f} % variance)')ax.set_ylabel(f'PC 2  ({v2:.1f} % variance)')ax.set_title('Figure 5. PCA with 95 % Confidence Ellipses',             fontsize=13, fontweight='bold')ax.axhline(0, color='gray', alpha=0.2)ax.axvline(0, color='gray', alpha=0.2)ax.legend(fontsize=9)ax.grid(True, alpha=0.15)plt.tight_layout()plt.show()

In [ ]:
# ── Figure 6: Discriminant Function Analysis ──────────────────────────fig, ax = plt.subplots(figsize=(10, 8))for g, c in GC.items():    m = dfa[GRP] == g    ax.scatter(dfa.loc[m, LD1], dfa.loc[m, LD2],               c=c, alpha=0.5, s=45, edgecolors='gray', lw=0.3, label=GS[g])ax.set_xlabel('DF 1')ax.set_ylabel('DF 2')ax.set_title('Figure 6. Discriminant Function Analysis', fontsize=13, fontweight='bold')ax.legend(fontsize=9)ax.grid(True, alpha=0.15)plt.tight_layout()plt.show()# Classification accuracyprint('\nDFA Classification Accuracy')for g in groups:    m  = dfa[GRP] == g    ok = (dfa.loc[m, 'Classification'] == 'Correct').sum()    t  = m.sum()    print(f'  {GS[g]:25s}  {ok}/{t} = {ok/t*100:.1f} %')

In [ ]:
# ── Figure 11: La/Sc vs Th/Sc Provenance Diagram ─────────────────────fig, ax = plt.subplots(figsize=(10, 8))for g, c in GC.items():    m = indices[GRP] == g    ax.scatter(indices.loc[m, ThSc], indices.loc[m, LaSc],               c=c, alpha=0.55, s=45, edgecolors='gray', lw=0.3, label=GS[g])ax.set_xlabel('Th/Sc')ax.set_ylabel('La/Sc')ax.set_title('Figure 11. La/Sc vs Th/Sc — Provenance Discrimination',             fontsize=13, fontweight='bold')ax.legend(fontsize=9)ax.grid(True, alpha=0.15)plt.tight_layout()plt.show()

## 8. Cluster Validation & Multi-Proxy Agreement

In [ ]:
# ── Figure 15: Silhouette Coefficient Distribution ────────────────────fig, ax = plt.subplots(figsize=(10, 7))for g, c in GC.items():    m = pca[GRP] == g    ax.hist(pca.loc[m, Sil].dropna(), bins=20, alpha=0.40,            color=c, label=GS[g], edgecolor='white', lw=0.5)ax.axvline(x=0.5, color='red', ls='--', alpha=0.6, label='Threshold 0.5')ax.set_xlabel('Silhouette Coefficient')ax.set_ylabel('Frequency')ax.set_title('Figure 15. Silhouette Score Distributions',             fontsize=13, fontweight='bold')ax.legend(fontsize=9)plt.tight_layout()plt.show()print(f'Overall Mean Silhouette: {pca[Sil].mean():.3f}')

In [ ]:
# ── Figure 13: Provenance Confidence Heatmap (pure matplotlib) ────────conf_levels = ['High', 'Moderate', 'Low']matrix = []for g in groups:    m = provenance[GRP_PROV] == g    total = m.sum()    row = [(provenance.loc[m, 'Confidence_Level'] == cl).sum() / total * 100           if total else 0 for cl in conf_levels]    matrix.append(row)matrix = np.array(matrix)fig, ax = plt.subplots(figsize=(8, 6))im = ax.imshow(matrix, cmap='YlOrRd', vmin=0, vmax=100, aspect='auto')ax.set_xticks(range(len(conf_levels)))ax.set_xticklabels(conf_levels, fontsize=10)ax.set_yticks(range(len(groups)))ax.set_yticklabels([GS[g] for g in groups], fontsize=10)for i in range(len(groups)):    for j in range(len(conf_levels)):        val = matrix[i, j]        colour = 'white' if val > 50 else 'black'        ax.text(j, i, f'{val:.1f}%', ha='center', va='center',                fontsize=9, fontweight='bold', color=colour)fig.colorbar(im, ax=ax, shrink=0.8, label='% of Assignments')ax.set_title('Figure 13. Provenance Confidence by Clay Source',             fontsize=13, fontweight='bold')plt.tight_layout()plt.show()# ── Table 5: Multi-Proxy Summary ──────────────────────────────────────print('\nTable 5. Provenance Assignment Summary')print(f'  {"Group":25s}  {"n":>5}  {"High%":>7}  {"Local%":>7}  {"Mean D²":>8}')print('  ' + '-' * 60)for g in groups:    m   = provenance[GRP_PROV] == g    t   = m.sum()    hi  = (provenance.loc[m, 'Confidence_Level'] == 'High').sum() / t * 100    loc = (provenance.loc[m, 'Local_vs_NonLocal'] == 'Local').sum() / t * 100    d2  = provenance.loc[m, 'Mahalanobis_D2'].mean()    print(f'  {GS[g]:25s}  {t:5d}  {hi:6.1f}%  {loc:6.1f}%  {d2:8.2f}')

## 9. Statistical Tests & Provenance Indices

In [ ]:
# ── One-Way ANOVA on Major Oxides + CIA + ICV ─────────────────────────print('One-Way ANOVA')print(f'  {"Variable":20s}  {"F":>10}  {"p-value":>12}  {"Sig."}')print('  ' + '-' * 55)test_cols = oxide_cols + [CIA_col, ICV_col]for v in test_cols:    src = xrf if v in xrf.columns else indices    grps = [src.loc[src[GRP] == g, v].dropna() for g in groups]    grps = [x for x in grps if len(x) > 0]    if len(grps) >= 2:        F, p = f_oneway(*grps)        sig  = 'p<0.001' if p < 0.001 else ('p<0.05' if p < 0.05 else 'NS')        print(f'  {v:20s}  {F:10.2f}  {p:12.2e}  {sig}')# ── Kruskal–Wallis on REE & Other Parameters ─────────────────────────print('\nKruskal–Wallis')print(f'  {"Variable":25s}  {"H":>10}  {"p-value":>12}')print('  ' + '-' * 55)kw_tests = [    ('Total_REE_ppm', ree),    ('LREE_HREE',     ree),    ('Eu_Eu_star',    ree),    ('Silhouette_Coeff', pca),]for param, df in kw_tests:    grps = [df.loc[df[GRP] == g, param].dropna() for g in groups]    grps = [x for x in grps if len(x) > 0]    if len(grps) >= 2:        H, p = kruskal(*grps)        print(f'  {param:25s}  {H:10.2f}  {p:12.2e}')# ── Provenance Indices Summary ────────────────────────────────────────print('\nProvenance Indices (mean ± σ)')idx_cols = [LaSc, ThSc, 'Cr_V', 'Al2O3_TiO2']hdr = f'  {"Group":25s}' + ''.join(f'  {c:>16s}' for c in idx_cols)print(hdr)print('  ' + '-' * (25 + 18 * len(idx_cols)))for g in groups:    m = indices[GRP] == g    parts = [f'{GS[g]:25s}']    for c in idx_cols:        v = indices.loc[m, c].dropna()        parts.append(f'{v.mean():.2f} ± {v.std():.2f}')    print('  ' + '  '.join(parts))

## 10. Clay Mineralogy (XRD)

In [ ]:
# ── Figure: XRD Mineralogy — Kaolinite vs Smectite ────────────────────fig, (a1, a2) = plt.subplots(1, 2, figsize=(16, 7))for g, c in GC.items():    m = xrd[GRP] == g    a1.scatter(xrd.loc[m, 'Kaolinite_%'], xrd.loc[m, 'Smectite_IS_%'],               c=c, alpha=0.5, s=40, edgecolors='gray', lw=0.3, label=GS[g])    a2.scatter(xrd.loc[m, 'Illite_%'], xrd.loc[m, 'Chlorite_%'],               c=c, alpha=0.5, s=40, edgecolors='gray', lw=0.3, label=GS[g])a1.set_xlabel('Kaolinite %'); a1.set_ylabel('Smectite/I-S %')a1.set_title('Kaolinite vs Smectite/I-S', fontweight='bold')a1.legend(fontsize=8); a1.grid(True, alpha=0.15)a2.set_xlabel('Illite %'); a2.set_ylabel('Chlorite %')a2.set_title('Illite vs Chlorite', fontweight='bold')a2.legend(fontsize=8); a2.grid(True, alpha=0.15)fig.suptitle('XRD Clay Mineralogy', fontsize=14, fontweight='bold', y=1.02)plt.tight_layout()plt.show()

## 11. Summary

In [ ]:
print('=' * 80)print('ANALYTICAL PIPELINE — EXECUTION SUMMARY')print('=' * 80)print(f'Sheets loaded       : {len(ALL)}')print(f'Total data cells    : {sum(d.shape[0]*d.shape[1] for d in ALL.values()):,}')print(f'Clay-source groups  : {len(groups)}')print(f'Specimens (xrf)     : {len(xrf):,}')print(f'REE analyses        : {len(ree):,}')print(f'PCA scores          : {len(pca):,}')print(f'DFA classifications : {len(dfa):,}')print(f'Mean Silhouette     : {pca[Sil].mean():.3f}')print(f'DFA overall accuracy: {(dfa["Classification"]=="Correct").mean()*100:.1f} %')print('=' * 80)print()print('Groups and sample counts:')for g in groups:    n = (xrf[GRP] == g).sum()    print(f'  {GS[g]:25s}  n = {n}')

---**Jackson, Majumder, Jaiswal, Patterson & Goswami (2026) | Editor: Sania Mughal**